# 面试题：两阶段推荐系统怎样设计召回、重排和评估？

## 可以直接复述的回答

两阶段推荐先用便宜的召回从全库找出几十到几千个候选，再用更丰富特征重排，不能让重排器扫描全部商品。召回阶段优化目标是相关商品能否进入候选集，重排阶段才优化第一名或列表质量，因此要分别报告 Recall@K 和 Hit@1。候选生成必须在截断 TopK 前完成库存、权限和已看过滤，否则无效商品会占满有限名额。教学实现可以用用户历史品类均值与商品向量做 cosine 召回，再用偏好、新鲜度、毛利和场景匹配线性重排。流行度是必要基线，它能揭示个性化是否真的提供增量。评估必须按截止时间构建用户画像，不能把未来购买写回历史特征。本题用 6 个用户、14 个商品和 6 个未来购买，输出完整候选漏斗并复现“先截断后过滤”的空候选失败。

## 真实案例

六位脱敏用户分别偏好跑步、摄影、咖啡、游戏、护肤和家居；商品含品类向量、流行度、新鲜度、毛利、库存与场景字段。数据为离线教学构造，只用于解释两阶段机制。

In [1]:
from pprint import pprint  # 导入结构化打印函数以展示候选漏斗
import numpy as np  # 导入 NumPy 以手算用户和商品向量相似度
categories = ["跑步", "摄影", "咖啡", "游戏", "护肤", "家居"]  # 定义六维可解释兴趣空间
def category_vector(category):  # 定义品类到 one-hot 向量的转换
    vector = [0.0] * len(categories)  # 创建六维零向量
    vector[categories.index(category)] = 1.0  # 在目标品类位置写入一
    return vector  # 返回可解释品类向量
items = [{"id": "I0", "标题": "热门会员礼包", "category": "通用", "vector": [0.4] * 6, "popularity": 100, "freshness": 0.7, "margin": 0.8, "stock": True, "scene": "all"}, {"id": "I1", "标题": "旧款跑鞋", "category": "跑步", "vector": category_vector("跑步"), "popularity": 70, "freshness": 0.2, "margin": 0.4, "stock": True, "scene": "outdoor"}, {"id": "I2", "标题": "新款竞速跑鞋", "category": "跑步", "vector": category_vector("跑步"), "popularity": 35, "freshness": 1.0, "margin": 0.7, "stock": True, "scene": "outdoor"}, {"id": "I2x", "标题": "断货限量跑鞋", "category": "跑步", "vector": category_vector("跑步"), "popularity": 90, "freshness": 1.0, "margin": 0.9, "stock": False, "scene": "outdoor"}, {"id": "I3", "标题": "入门相机", "category": "摄影", "vector": category_vector("摄影"), "popularity": 65, "freshness": 0.3, "margin": 0.4, "stock": True, "scene": "travel"}, {"id": "I4", "标题": "旅行变焦镜头", "category": "摄影", "vector": category_vector("摄影"), "popularity": 30, "freshness": 0.9, "margin": 0.8, "stock": True, "scene": "travel"}, {"id": "I5", "标题": "旧款咖啡机", "category": "咖啡", "vector": category_vector("咖啡"), "popularity": 60, "freshness": 0.2, "margin": 0.5, "stock": True, "scene": "home"}, {"id": "I6", "标题": "精密磨豆机", "category": "咖啡", "vector": category_vector("咖啡"), "popularity": 28, "freshness": 0.9, "margin": 0.8, "stock": True, "scene": "home"}, {"id": "I7", "标题": "旧款游戏鼠标", "category": "游戏", "vector": category_vector("游戏"), "popularity": 58, "freshness": 0.2, "margin": 0.4, "stock": True, "scene": "desktop"}, {"id": "I8", "标题": "低延迟机械键盘", "category": "游戏", "vector": category_vector("游戏"), "popularity": 27, "freshness": 1.0, "margin": 0.7, "stock": True, "scene": "desktop"}, {"id": "I9", "标题": "基础洁面", "category": "护肤", "vector": category_vector("护肤"), "popularity": 55, "freshness": 0.3, "margin": 0.5, "stock": True, "scene": "morning"}, {"id": "I10", "标题": "通勤防晒", "category": "护肤", "vector": category_vector("护肤"), "popularity": 25, "freshness": 0.9, "margin": 0.7, "stock": True, "scene": "morning"}, {"id": "I11", "标题": "旧款台灯", "category": "家居", "vector": category_vector("家居"), "popularity": 50, "freshness": 0.2, "margin": 0.4, "stock": True, "scene": "night"}, {"id": "I12", "标题": "护眼阅读灯", "category": "家居", "vector": category_vector("家居"), "popularity": 24, "freshness": 1.0, "margin": 0.8, "stock": True, "scene": "night"}]  # 构造十四个含库存和业务特征的商品
users = [{"id": "U1", "history": ["I1"], "scene": "outdoor", "gold": "I2"}, {"id": "U2", "history": ["I3"], "scene": "travel", "gold": "I4"}, {"id": "U3", "history": ["I5"], "scene": "home", "gold": "I6"}, {"id": "U4", "history": ["I7"], "scene": "desktop", "gold": "I8"}, {"id": "U5", "history": ["I9"], "scene": "morning", "gold": "I10"}, {"id": "U6", "history": ["I11"], "scene": "night", "gold": "I12"}]  # 构造六个截止日前画像及未来购买标签
items[3]["id"] = "I1x"  # 调整断货跑鞋编号使错误流程的前两名都被无效商品占据
item_by_id = {item["id"]: item for item in items}  # 建立商品 ID 到完整字段的索引
print("商品与库存输入预览：")  # 输出真实商品标题
pprint([{key: item[key] for key in ["id", "标题", "category", "popularity", "freshness", "margin", "stock", "scene"]} for item in items])  # 展示召回和重排字段
print("用户历史、场景与未来购买：")  # 输出评估样本标题
pprint(users)  # 展示六个用户的时间切分信息

商品与库存输入预览：
[{'category': '通用',
  'freshness': 0.7,
  'id': 'I0',
  'margin': 0.8,
  'popularity': 100,
  'scene': 'all',
  'stock': True,
  '标题': '热门会员礼包'},
 {'category': '跑步',
  'freshness': 0.2,
  'id': 'I1',
  'margin': 0.4,
  'popularity': 70,
  'scene': 'outdoor',
  'stock': True,
  '标题': '旧款跑鞋'},
 {'category': '跑步',
  'freshness': 1.0,
  'id': 'I2',
  'margin': 0.7,
  'popularity': 35,
  'scene': 'outdoor',
  'stock': True,
  '标题': '新款竞速跑鞋'},
 {'category': '跑步',
  'freshness': 1.0,
  'id': 'I1x',
  'margin': 0.9,
  'popularity': 90,
  'scene': 'outdoor',
  'stock': False,
  '标题': '断货限量跑鞋'},
 {'category': '摄影',
  'freshness': 0.3,
  'id': 'I3',
  'margin': 0.4,
  'popularity': 65,
  'scene': 'travel',
  'stock': True,
  '标题': '入门相机'},
 {'category': '摄影',
  'freshness': 0.9,
  'id': 'I4',
  'margin': 0.8,
  'popularity': 30,
  'scene': 'travel',
  'stock': True,
  '标题': '旅行变焦镜头'},
 {'category': '咖啡',
  'freshness': 0.2,
  'id': 'I5',
  'margin': 0.5,
  'popularity': 60,
  'scene': 

## Baseline / 基线：屏蔽已购后的全局流行度

流行度基线会把热门会员礼包推给所有用户。它遵守库存和已购过滤，因此与两阶段方案使用同一候选全集，但没有个性化。

In [2]:
def eligible_items(user):  # 定义库存与已购商品门禁
    return [item for item in items if item["stock"] and item["id"] not in user["history"]]  # 返回当前用户可以推荐的商品
def popularity_ranking(user):  # 定义全局流行度基线
    candidates = eligible_items(user)  # 先执行同口径候选门禁
    return sorted(candidates, key=lambda item: (-item["popularity"], item["id"]))  # 按流行度和商品 ID 稳定排序
baseline_rankings = {user["id"]: popularity_ranking(user) for user in users}  # 对六位用户运行流行度基线
baseline_hits = sum(baseline_rankings[user["id"]][0]["id"] == user["gold"] for user in users)  # 统计基线 Hit@1
print("Popularity Baseline Top3：")  # 输出基线结果标题
pprint([{"用户": user["id"], "gold": user["gold"], "Top3": [item["id"] for item in baseline_rankings[user["id"]][:3]]} for user in users])  # 展示逐用户基线候选
print(f"Popularity Hit@1={baseline_hits}/{len(users)}")  # 输出基线汇总指标

Popularity Baseline Top3：
[{'Top3': ['I0', 'I3', 'I5'], 'gold': 'I2', '用户': 'U1'},
 {'Top3': ['I0', 'I1', 'I5'], 'gold': 'I4', '用户': 'U2'},
 {'Top3': ['I0', 'I1', 'I3'], 'gold': 'I6', '用户': 'U3'},
 {'Top3': ['I0', 'I1', 'I3'], 'gold': 'I8', '用户': 'U4'},
 {'Top3': ['I0', 'I1', 'I3'], 'gold': 'I10', '用户': 'U5'},
 {'Top3': ['I0', 'I1', 'I3'], 'gold': 'I12', '用户': 'U6'}]
Popularity Hit@1=0/6


## 第一阶段：由截止日前历史构造用户向量并召回

用户向量仅平均 history 中商品品类向量，绝不读取 gold。召回先过滤库存与已购，再按 cosine 取 Top5；这保证无效商品不占有限候选位。

In [3]:
def cosine(left, right):  # 定义 NumPy 余弦相似度
    left_vector = np.asarray(left, dtype=np.float64)  # 转换用户向量为浮点数组
    right_vector = np.asarray(right, dtype=np.float64)  # 转换商品向量为浮点数组
    denominator = np.linalg.norm(left_vector) * np.linalg.norm(right_vector)  # 计算范数乘积
    return float(left_vector @ right_vector / denominator) if denominator else 0.0  # 返回归一化点积并保护零向量
def user_profile(user):  # 定义只使用截止日前历史的画像构造
    history_vectors = [np.asarray(item_by_id[item_id]["vector"], dtype=np.float64) for item_id in user["history"]]  # 读取历史商品品类向量
    return np.mean(np.stack(history_vectors), axis=0)  # 对历史向量求均值得到用户兴趣
def retrieve_candidates(user, top_k=5):  # 定义低成本个性化召回阶段
    profile = user_profile(user)  # 计算不含未来标签的用户画像
    candidates = eligible_items(user)  # 在截断前完成库存和已购过滤
    rows = [{"item_id": item["id"], "retrieval_score": cosine(profile, item["vector"])} for item in candidates]  # 实算全部合格商品语义分
    return sorted(rows, key=lambda row: (-row["retrieval_score"], row["item_id"]))[:top_k]  # 截取召回前五名
retrieved = {user["id"]: retrieve_candidates(user) for user in users}  # 为六位用户生成候选集
candidate_recall = sum(user["gold"] in [row["item_id"] for row in retrieved[user["id"]]] for user in users) / len(users)  # 计算未来购买进入 Top5 的比例
print("第一阶段逐用户 Top5 候选：")  # 输出召回结果标题
pprint([{"用户": user["id"], "gold": user["gold"], "候选": retrieved[user["id"]]} for user in users])  # 展示召回分与候选覆盖
print(f"Candidate Recall@5={candidate_recall:.3f}")  # 输出阶段一独立指标

第一阶段逐用户 Top5 候选：
[{'gold': 'I2',
  '候选': [{'item_id': 'I2', 'retrieval_score': 1.0},
         {'item_id': 'I0', 'retrieval_score': 0.408248290463863},
         {'item_id': 'I10', 'retrieval_score': 0.0},
         {'item_id': 'I11', 'retrieval_score': 0.0},
         {'item_id': 'I12', 'retrieval_score': 0.0}],
  '用户': 'U1'},
 {'gold': 'I4',
  '候选': [{'item_id': 'I4', 'retrieval_score': 1.0},
         {'item_id': 'I0', 'retrieval_score': 0.408248290463863},
         {'item_id': 'I1', 'retrieval_score': 0.0},
         {'item_id': 'I10', 'retrieval_score': 0.0},
         {'item_id': 'I11', 'retrieval_score': 0.0}],
  '用户': 'U2'},
 {'gold': 'I6',
  '候选': [{'item_id': 'I6', 'retrieval_score': 1.0},
         {'item_id': 'I0', 'retrieval_score': 0.408248290463863},
         {'item_id': 'I1', 'retrieval_score': 0.0},
         {'item_id': 'I10', 'retrieval_score': 0.0},
         {'item_id': 'I11', 'retrieval_score': 0.0}],
  '用户': 'U3'},
 {'gold': 'I8',
  '候选': [{'item_id': 'I8', 'retrieval_scor

## 第二阶段：用丰富业务特征重排

重排器只处理 Top5。分数包含召回相似度、新鲜度、毛利和当前场景匹配；这里用透明人工权重展示机制，线上权重应由带时间切分的训练数据学习。

In [4]:
def rerank(user, candidates):  # 定义只处理召回候选的轻量重排器
    rows = []  # 创建候选特征和最终分账本
    for candidate in candidates:  # 遍历第一阶段最多五个候选
        item = item_by_id[candidate["item_id"]]  # 读取候选完整业务字段
        scene_match = 1.0 if item["scene"] in {user["scene"], "all"} else 0.0  # 计算当前使用场景是否匹配
        rerank_score = 0.55 * candidate["retrieval_score"] + 0.20 * item["freshness"] + 0.15 * item["margin"] + 0.10 * scene_match  # 组合个性化与业务特征
        rows.append({"item_id": item["id"], "召回分": round(candidate["retrieval_score"], 4), "新鲜度": item["freshness"], "毛利": item["margin"], "场景匹配": scene_match, "重排分": round(rerank_score, 4)})  # 保存逐候选可审计特征
    return sorted(rows, key=lambda row: (-row["重排分"], row["item_id"]))  # 返回最终推荐排名
reranked = {user["id"]: rerank(user, retrieved[user["id"]]) for user in users}  # 对六个候选集执行第二阶段
print("U1 的召回到重排候选漏斗：")  # 输出关键中间量标题
pprint({"用户画像": user_profile(users[0]).tolist(), "召回": retrieved["U1"], "重排": reranked["U1"]})  # 展示阶段间候选和特征变化

U1 的召回到重排候选漏斗：
{'召回': [{'item_id': 'I2', 'retrieval_score': 1.0},
        {'item_id': 'I0', 'retrieval_score': 0.408248290463863},
        {'item_id': 'I10', 'retrieval_score': 0.0},
        {'item_id': 'I11', 'retrieval_score': 0.0},
        {'item_id': 'I12', 'retrieval_score': 0.0}],
 '用户画像': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 '重排': [{'item_id': 'I2',
         '召回分': 1.0,
         '场景匹配': 1.0,
         '新鲜度': 1.0,
         '毛利': 0.7,
         '重排分': 0.955},
        {'item_id': 'I0',
         '召回分': 0.4082,
         '场景匹配': 1.0,
         '新鲜度': 0.7,
         '毛利': 0.8,
         '重排分': 0.5845},
        {'item_id': 'I12',
         '召回分': 0.0,
         '场景匹配': 0.0,
         '新鲜度': 1.0,
         '毛利': 0.8,
         '重排分': 0.32},
        {'item_id': 'I10',
         '召回分': 0.0,
         '场景匹配': 0.0,
         '新鲜度': 0.9,
         '毛利': 0.7,
         '重排分': 0.285},
        {'item_id': 'I11',
         '召回分': 0.0,
         '场景匹配': 0.0,
         '新鲜度': 0.2,
         '毛利': 0.4,
         '重排分': 0.1

## 逐用户结果与结果解读

召回负责让未来购买进入 Top5，重排再让新款、毛利合理且场景匹配的商品到第一。通用礼包仍可能进入候选，但个性化相似度较低，不再支配所有用户。

In [5]:
result_rows = []  # 创建逐用户两阶段结果表
for user in users:  # 遍历六位评估用户
    baseline_top = baseline_rankings[user["id"]][0]["id"]  # 读取流行度基线第一名
    final_top = reranked[user["id"]][0]["item_id"]  # 读取两阶段重排第一名
    result_rows.append({"用户": user["id"], "gold": user["gold"], "Popularity Top1": baseline_top, "候选含gold": user["gold"] in [row["item_id"] for row in retrieved[user["id"]]], "TwoStage Top1": final_top, "正确": final_top == user["gold"]})  # 保存召回和重排两级决策
two_stage_hits = sum(row["正确"] for row in result_rows)  # 统计最终 Hit@1
print("逐用户推荐结果：")  # 输出结果表标题
pprint(result_rows)  # 展示每个用户的候选覆盖和最终推荐
print(f"Hit@1 从 {baseline_hits}/{len(users)} 提升到 {two_stage_hits}/{len(users)}，Recall@5={candidate_recall:.3f}")  # 输出同数据两阶段指标

逐用户推荐结果：
[{'Popularity Top1': 'I0',
  'TwoStage Top1': 'I2',
  'gold': 'I2',
  '候选含gold': True,
  '正确': True,
  '用户': 'U1'},
 {'Popularity Top1': 'I0',
  'TwoStage Top1': 'I4',
  'gold': 'I4',
  '候选含gold': True,
  '正确': True,
  '用户': 'U2'},
 {'Popularity Top1': 'I0',
  'TwoStage Top1': 'I6',
  'gold': 'I6',
  '候选含gold': True,
  '正确': True,
  '用户': 'U3'},
 {'Popularity Top1': 'I0',
  'TwoStage Top1': 'I8',
  'gold': 'I8',
  '候选含gold': True,
  '正确': True,
  '用户': 'U4'},
 {'Popularity Top1': 'I0',
  'TwoStage Top1': 'I10',
  'gold': 'I10',
  '候选含gold': True,
  '正确': True,
  '用户': 'U5'},
 {'Popularity Top1': 'I0',
  'TwoStage Top1': 'I12',
  'gold': 'I12',
  '候选含gold': True,
  '正确': True,
  '用户': 'U6'}]
Hit@1 从 0/6 提升到 6/6，Recall@5=1.000


## 失败案例：先截断 TopK，再过滤无效商品

U1 的品类最相似商品包括已购旧鞋、断货限量鞋和目标新鞋。错误流程先取 Top2，随后过滤会把两个位置都删除，得到空候选；正确流程先过滤再截断，可以保留新鞋。

In [6]:
def unsafe_retrieve_then_filter(user, top_k=2):  # 定义先截断再过滤的错误候选流程
    profile = user_profile(user)  # 计算相同的截止日前用户画像
    all_rows = [{"item_id": item["id"], "score": cosine(profile, item["vector"])} for item in items]  # 错误地让已购和断货商品进入排序
    top_rows = sorted(all_rows, key=lambda row: (-row["score"], row["item_id"]))[:top_k]  # 在门禁前提前截断有限名额
    return [row for row in top_rows if item_by_id[row["item_id"]]["stock"] and row["item_id"] not in user["history"]]  # 截断后才移除无效商品
unsafe_candidates = unsafe_retrieve_then_filter(users[0], top_k=2)  # 复现 U1 候选被无效商品占满
safe_candidates = retrieve_candidates(users[0], top_k=2)  # 使用过滤在前的正确召回流程
print("失败案例：先 Top2 后过滤的候选", unsafe_candidates)  # 展示无有效候选的错误结果
print("修正：先过滤再 Top2 的候选", safe_candidates)  # 展示目标新鞋被保留

失败案例：先 Top2 后过滤的候选 []
修正：先过滤再 Top2 的候选 [{'item_id': 'I2', 'retrieval_score': 1.0}, {'item_id': 'I0', 'retrieval_score': 0.408248290463863}]


## 生产差距

线上召回通常包含协同过滤、向量 ANN、规则与探索多个通道，并需要去重和配额；重排模型还会使用实时上下文、价格、转化和长期价值。所有特征必须按请求时间回放以防未来泄漏，还要监控候选 Recall、过滤损耗、覆盖率、多样性、库存新鲜度、延迟和在线实验结果。

In [7]:
assert len(users) == 6 and len(items) == 14  # 验证案例包含六位用户和十四个商品
assert candidate_recall == 1.0  # 验证第一阶段候选覆盖全部未来购买
assert two_stage_hits > baseline_hits  # 验证两阶段方案优于同候选门禁流行度基线
assert two_stage_hits == len(users)  # 验证六位用户的未来购买均排在第一名
assert unsafe_candidates == []  # 验证先截断后过滤会产生空候选失败
assert safe_candidates[0]["item_id"] == "I2"  # 验证前置过滤保留目标新款跑鞋
print("最小回归测试通过：候选召回、业务重排、阶段评估与过滤顺序均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：候选召回、业务重排、阶段评估与过滤顺序均满足预期
